## Fine-tuning эмбеддингов

In [1]:
%load_ext autoreload
%autoreload 3

In [2]:
from torch.optim import Adam
from torch.utils.data import DataLoader

from sneakers_hse.data.utils.s3_tools import S3Client

import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path
import os

In [3]:
PROJECT_ROOT_PATH = Path(os.getenv('PROJECT_ROOT_PATH'))
# s3 = S3Client(aws_access_key_id=os.getenv("AWS_ACCESS_KEY"),
#               aws_secret_access_key=os.getenv("AWS_SECRET_KEY"))
# s3._download_one(
#     bucket_name='sneakers-hse-images-test',
#     s3_key='dinov2/embeddings.parquet.gzip',
#     local_path=str(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings.parquet.gzip')
# )
# s3._download_one(
#     bucket_name='sneakers-hse-images-test',
#     s3_key='dinov2/embeddings_augmented_train.parquet.gzip',
#     local_path=str(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings_augmented_train.parquet.gzip')
# )

In [4]:
df = pl.read_parquet(PROJECT_ROOT_PATH / "data/04_dinov2_embeddings.parquet.gzip")

embeddings = np.stack(df["embedding"].to_list()).astype("float32")
labels = np.array(df["class"].to_list())
paths = df["path"].to_list()
df['class'].value_counts()
train_df_pre = pl.read_csv(PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/train_images.csv')
test_df = pl.read_csv(PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/test_images.csv')

train_df, val_df = train_test_split(
    train_df_pre,
    test_size=0.2,
    stratify=train_df_pre["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)
sql = pl.SQLContext()
sql.register_globals()
df = sql.execute(
    '''
select
    df.*
    , case when train_df.path is not null then 'train'
            when val_df.path is not null then 'val'
            when test_df.path is not null then 'test'
            end as sample_part
from df
left join train_df using(path)
left join val_df using(path)
left join test_df using(path)
'''
).collect()
df

,path,sneaker_class,corrupted_flg
i64,str,str,i64
2288,"""Nike Кеды Dunk Low Retro/cloth…","""Nike Кеды Dunk Low Retro""",0
10855,"""PUMA Кроссовки Puma Morphic/or…","""PUMA Кроссовки Puma Morphic""",0
7477,"""Kari Кроссовки/clothing_0_190.…","""Kari Кроссовки""",0
4230,"""Reebok Кроссовки CLASSIC LEATH…","""Reebok Кроссовки CLASSIC LEATH…",0
2188,"""Nike Кеды Dunk Low Retro/cloth…","""Nike Кеды Dunk Low Retro""",0


(8772, 4)

,path,sneaker_class,corrupted_flg
i64,str,str,i64
7639,"""Vans Кеды Knu Skool/clothing_0…","""Vans Кеды Knu Skool""",0
2591,"""Reebok Кроссовки CLASSIC NYLON…","""Reebok Кроссовки CLASSIC NYLON""",0
5027,"""Nike Кроссовки AIR MAX 90/clot…","""Nike Кроссовки AIR MAX 90""",0
2762,"""Under Armour Кроссовки UA CHAR…","""Under Armour Кроссовки UA CHAR…",0
240,"""Vans Кеды Upland/clothing_0_62…","""Vans Кеды Upland""",0


(2193, 4)

,path,sneaker_class,corrupted_flg
i64,str,str,i64
14,"""Vans Кеды Upland/clothing_0_16…","""Vans Кеды Upland""",0
32,"""Vans Кеды Upland/clothing_0_21…","""Vans Кеды Upland""",0
44,"""Vans Кеды Upland/orig_216_real…","""Vans Кеды Upland""",0
80,"""Vans Кеды Upland/shoe_3_100_re…","""Vans Кеды Upland""",0
87,"""Vans Кеды Upland/clothing_0_27…","""Vans Кеды Upland""",0


(300, 4)

path,class,embedding,sample_part
str,str,list[f64],str
"""Vans Кеды Upland/clothing_0_26…","""Vans Кеды Upland""","[-1.930393, -0.157552, … 0.300103]","""train"""
"""Vans Кеды Upland/clothing_0_57…","""Vans Кеды Upland""","[-1.749115, -1.480964, … -0.966375]","""train"""
"""Vans Кеды Upland/orig_45.jpeg""","""Vans Кеды Upland""","[0.503809, -0.948843, … -3.250608]","""train"""
"""Vans Кеды Upland/clothing_0_0.…","""Vans Кеды Upland""","[-1.441995, -2.107342, … -0.808429]","""train"""
"""Vans Кеды Upland/clothing_0_23…","""Vans Кеды Upland""","[-0.845633, -2.200308, … -1.242736]","""train"""
…,…,…,…
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[0.131167, -0.318168, … -0.66955]","""train"""
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.267569, -3.275375, … -2.853185]","""train"""
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.495174, -1.613851, … -1.374945]","""train"""


In [5]:
train_df = df.filter(pl.col('sample_part') == 'train')
val_df = df.filter(pl.col('sample_part') == 'val')
test_df = df.filter(pl.col('sample_part') == 'test')
train_df

path,class,embedding,sample_part
str,str,list[f64],str
"""Vans Кеды Upland/clothing_0_26…","""Vans Кеды Upland""","[-1.930393, -0.157552, … 0.300103]","""train"""
"""Vans Кеды Upland/clothing_0_57…","""Vans Кеды Upland""","[-1.749115, -1.480964, … -0.966375]","""train"""
"""Vans Кеды Upland/orig_45.jpeg""","""Vans Кеды Upland""","[0.503809, -0.948843, … -3.250608]","""train"""
"""Vans Кеды Upland/clothing_0_0.…","""Vans Кеды Upland""","[-1.441995, -2.107342, … -0.808429]","""train"""
"""Vans Кеды Upland/clothing_0_23…","""Vans Кеды Upland""","[-0.845633, -2.200308, … -1.242736]","""train"""
…,…,…,…
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[-1.853474, -1.394531, … -1.656443]","""train"""
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[0.131167, -0.318168, … -0.66955]","""train"""
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.267569, -3.275375, … -2.853185]","""train"""


In [6]:
aug_df = pl.read_parquet(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings_augmented_train.parquet.gzip')

train_df = pl.concat([
    train_df,
    aug_df.with_columns(pl.lit('train').alias('sample_part'))
])

print(f"train: {len(train_df)} ({len(train_df) - len(aug_df)} orig + {len(aug_df)} aug), val: {len(val_df)}, test: {len(test_df)}")

train: 41667 (8772 orig + 32895 aug), val: 2193, test: 300


## Обучаю модель

In [7]:
from sneakers_hse.model.triplet_loss import EmbeddingDataset, LitTripletModel
from pytorch_metric_learning.samplers import MPerClassSampler

train_dataset = EmbeddingDataset(train_df)
sampler = MPerClassSampler(labels=train_dataset.labels, m=4, batch_size=64, length_before_new_iter=1_000)
train_dataloader = DataLoader(train_dataset, batch_size=64, sampler=sampler)

val_dataset = EmbeddingDataset(val_df)
val_dataloader = DataLoader(val_dataset, batch_size=64)

In [8]:
for x, y in train_dataloader:
    print(x.shape)
    break

torch.Size([64, 768])


In [9]:
from pytorch_metric_learning import losses, miners

loss_fn = losses.TripletMarginLoss(margin=0.2)

miner = miners.TripletMarginMiner(
    margin=0.2,
    type_of_triplets="semihard"
)

In [10]:
model = LitTripletModel(input_dim=768, embedding_dim=768, lr=1e-3)
model

LitTripletModel(
  (model): Sequential(
    (0): Linear(in_features=768, out_features=1024, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=1024, out_features=1024, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=1024, out_features=768, bias=True)
  )
  (loss_fn): TripletMarginLoss(
    (distance): CosineSimilarity()
    (reducer): AvgNonZeroReducer()
  )
  (miner): TripletMarginMiner(
    (distance): CosineSimilarity()
  )
)

In [11]:
import pytorch_lightning as L
from pytorch_lightning.loggers import TensorBoardLogger
from sneakers_hse.model.triplet_loss import LitTripletModel

model = LitTripletModel(input_dim=768, embedding_dim=768, lr=1e-3)

logger = TensorBoardLogger(str(PROJECT_ROOT_PATH / "tb_logs"), name="triplet_loss")

trainer = L.Trainer(max_epochs=50, logger=logger)
trainer.fit(model, train_dataloader, val_dataloader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ Sequential         │  2.6 M │ train │     0 │
│ 1 │ loss_fn │ TripletMarginLoss  │      0 │ train │     0 │
│ 2 │ miner   │ TripletMarginMiner │      0 │ train │     0 │
└───┴─────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 2.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.6 M                                                                                                
Total estimated model params size (MB): 10                                                                         
Modules in train mode: 12                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/opt/miniconda3/envs/sneakers-hse/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/opt/miniconda3/envs/sneakers-hse/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.
py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of
the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

/opt/miniconda3/envs/sneakers-hse/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.
py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value 
of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

/opt/miniconda3/envs/sneakers-hse/lib/python3.11/site-packages/pytorch_lightning/loops/fit_loop.py:317: The number 
of training batches (15) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for 
log_every_n_steps if you want to see logs for the training epoch.

`Trainer.fit` stopped: `max_epochs=50` reached.


In [12]:
import chromadb
from sneakers_hse.metrics import get_neighbors
from tqdm.notebook import tqdm
import torch

device = "mps" if torch.mps.is_available() else "cpu"
model.eval()
model.to(device)

def encode(df_part, label=""):
    loader = DataLoader(EmbeddingDataset(df_part), batch_size=64)
    parts = []
    for x, _ in tqdm(loader, desc=f"Encoding {label}", leave=False):
        with torch.no_grad():
            parts.append(model(x.to(device)).cpu().numpy())
    return np.vstack(parts)

# Кодируем только оригинальные сплиты (без аугментированных)
orig_train_df = df.filter(pl.col("sample_part") == "train")
splits = {"train": orig_train_df, "val": val_df, "test": test_df}

all_embeddings = np.vstack([encode(df_part, name) for name, df_part in splits.items()])
all_labels = np.array(
    orig_train_df["class"].to_list() + val_df["class"].to_list() + test_df["class"].to_list()
)

# Добавляем в одну коллекцию
client = chromadb.Client()
collection = client.get_or_create_collection("triplet_eval", metadata={"hnsw:space": "cosine"})
for i in range(0, len(all_embeddings), 5000):
    collection.add(
        ids=[str(j) for j in range(i, min(i + 5000, len(all_embeddings)))],
        embeddings=all_embeddings[i:i+5000],
        metadatas=[{"class": c} for c in all_labels[i:i+5000].tolist()]
    )

neighbors = get_neighbors(collection, all_embeddings, k=10)
client.delete_collection("triplet_eval")

# Считаем метрики по сплитам
ks = [1, 5, 10]
rows = []
offset = 0
for name, df_part in splits.items():
    n = len(df_part)
    part_labels = all_labels[offset:offset+n]
    part_neighbors = neighbors[offset:offset+n]
    offset += n

    row = {"split": name, "n": n}
    for k in ks:
        hits = np.mean([part_labels[i] in part_neighbors[i][:k] for i in range(n)])
        precision = np.mean([(part_neighbors[i][:k] == part_labels[i]).sum() / k for i in range(n)])
        row[f"Hit@{k}"] = round(float(hits), 4)
        row[f"Precision@{k}"] = round(float(precision), 4)
    rows.append(row)

display(pl.DataFrame(rows))

Encoding train:   0%|          | 0/138 [00:00<?, ?it/s]

Encoding val:   0%|          | 0/35 [00:00<?, ?it/s]

Encoding test:   0%|          | 0/5 [00:00<?, ?it/s]

split,n,Hit@1,Precision@1,Hit@5,Precision@5,Hit@10,Precision@10
str,i64,f64,f64,f64,f64,f64,f64
"""train""",8772,0.6065,0.6065,0.8356,0.5158,0.8939,0.4704
"""val""",2193,0.6215,0.6215,0.8404,0.5144,0.9042,0.4689
"""test""",300,0.3767,0.3767,0.6133,0.2747,0.7133,0.2467
